## Assembling DR samples

Re-pairing the reads after filtering with fastq-screen

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-repair_mcav-%j.out  # %j = job ID

module load conda/latest
conda activate assembly

SAMPLENAME="mcav"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt" 
READSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR"
OUTDIR="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}/repaired"
mkdir -p "$OUTDIR"

#Using repair.sh script from:https://jgi.doe.gov/data-and-tools/software-tools/bbtools/bb-tools-user-guide/repair-guide/
while IFS= read -r SAMPLEID; do

repair.sh in1=$READSPATH/"${SAMPLEID}"_host_removed_R1.tagged_filter.fastq.gz in2=$READSPATH/"${SAMPLEID}"_host_removed_R2.tagged_filter.fastq.gz \
out1=$OUTDIR/"${SAMPLEID}"_R1_ready.fastq.gz out2=$OUTDIR/"${SAMPLEID}"_R2_ready.fastq.gz \
outs=$OUTDIR/"${SAMPLEID}"singletons.fq repair;
 if [ $? -eq 0 ]; then
        echo "repair completed successfully for sample: $SAMPLEID"
    else
        echo "repair encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate
echo "Repair: All samples processed successfully."

# JOB-ID: 49190145
# bash script file name: bbtools_repair.sh

did for the others \
dcyl: 49195043 \
ssid: 51633803 \
dlab: 51634024 \
mmea: 51634174 \
pstr: 51634224


In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 5:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-cat-%j.out  # %j = job ID

module load conda/latest
conda activate assembly

SAMPLENAME="mcav"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt" 
READSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/${SAMPLENAME}/repaired"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
WORKDIR="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered"

# CONCATETATE all f and r seqs into single file (1 for f, 1 for r)
# Read the sample IDs from the file
while IFS= read -r SAMPLEID; do
    # Construct the file paths for forward and reverse reads
    FORWARD_READ="$READSPATH/${SAMPLEID}_R1_ready.fastq.gz"
    REVERSE_READ="$READSPATH/${SAMPLEID}_R2_ready.fastq.gz"

    # Check if the files exist before concatenating
    if [ -e "$FORWARD_READ" ]; then
        cat "$FORWARD_READ" >> "$WORKDIR/DR_${SAMPLENAME}_reads_R1.fastq.gz"
    else
        echo "Forward read file not found for sample $SAMPLEID"
    fi

    if [ -e "$REVERSE_READ" ]; then
        cat "$REVERSE_READ" >> "$WORKDIR/DR_${SAMPLENAME}_reads_R2.fastq.gz"
    else
        echo "Reverse read file not found for sample $SAMPLEID"
    fi
done < "$LISTPATH/${SAMPLELIST}"

conda deactivate
echo "Concatenation completed!"

# JOB-ID: 51850117
# bash script file name: DR_concat.sh

job IDs for the others: \
ssid: 51850118 \
dcyl: 51850141 \
dlab: 51850142 \
mmea: 51850143 \
pstr: 51850144

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long # extend time limit to longer than 2 days
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-assembly-mcav-%j.out  # %j = job ID

module load conda/latest
conda activate assembly

SAMPLENAME="mcav"
WORKDIR="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered
OUTDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies"
mkdir -p $OUTDIR

# ASSEMBLE reads into contigs (contiguous sequence - joins them together based on read overlap, and ensures there are no gaps
megahit --presets meta-large \
-1 "$WORKDIR"/DR_"$SAMPLENAME"_reads_R1.fastq.gz \
-2 "$WORKDIR"/DR_"$SAMPLENAME"_reads_R2.fastq.gz \
-o $OUTDIR/megahit_assembly_"$SAMPLENAME"
#this one has to make the directory; will fail if it already exists

conda deactivate
echo "Assembly completed!"

# JOB-ID: 51865143
# bash script file name: DR_assemble_mcav.sh

job IDs for the others: \
ssid:  51865144 \
dcyl:  51865145 \
dlab:  51865147 \
mmea:  51865149 \
pstr:  51865150

### Mapping

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long # extend time limit to longer than 2 days
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-mapping_mcav-%j.out  # %j = job ID

module load conda/latest
conda activate anvio-8

SAMPLENAME="mcav"
READSPATH="/scratch3/workspace/nikea_ulrich_uml_edu-data_download/DR_final_filtered/mcav/repaired"
CONTIGPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies/megahit_assembly_${SAMPLENAME}"
CONTIGFILE="final.contigs.fa"
WORKPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping"
mkdir -p "$WORKPATH"
XTRAFILES="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping/sams"
mkdir -p "$XTRAFILES"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="DR_${SAMPLENAME}_sampleids.txt" 
 
anvi-script-reformat-fasta $CONTIGPATH/$CONTIGFILE -o $WORKPATH/"${SAMPLENAME}.contigs-fixed.fsa" -l 1000 --simplify-names --report-file $WORKPATH/"${SAMPLENAME}"_contig-rename-report.txt

#fixes deflines (filters contigs and reformats so naming is cleaner)
#filtering seq length 1000bp...need to play around with filtering based on bp length
#deflines = sequence definition line. comes directly before its associated sequence in a fasta file


FIXEDCON="${SAMPLENAME}.contigs-fixed.fsa"

cd $WORKPATH
#this builds an index of your contigs, which only needs to happen once
bowtie2-build $FIXEDCON "$SAMPLENAME"_contigs
# will not accept path before contigs file - must be in the correct dir 

while IFS= read -r SAMPLEID; do
    #align reads to your contigs and collects that in a .sam file
    bowtie2 --threads 11 -x "$SAMPLENAME"_contigs -1 $READSPATH/"${SAMPLEID}"_R1_ready.fastq.gz -2 $READSPATH/"${SAMPLEID}"_R2_ready.fastq.gz -S $XTRAFILES/"${SAMPLEID}".sam
    #make sure to point it to the index not the FIXEDCON file (-x parameter)
    
    #converts your sam file to a bam file, but its neither sorted nor indexed, so we use an Anvi'O script to do so:
    samtools view -F 4 -b -S $XTRAFILES/"${SAMPLEID}".sam -o $WORKPATH/"${SAMPLEID}"-RAW.bam
   
    #index and sort your bam file
    anvi-init-bam $WORKPATH/"${SAMPLEID}"-RAW.bam -o $WORKPATH/"${SAMPLEID}".bam
    
    rm $WORKPATH/"${SAMPLEID}"-RAW.bam
done < "$LISTPATH/${SAMPLELIST}"
echo "Mapping success!"

#JOB ID: 51971152
#bash script: DR_mapping_mcav.sh

job IDs for others: \
ssid: 51971154 \
dcyl: 51971156 \
dlab: 51971157 \
mmea: 51971158 \
pstr: 51971160

In [ ]:
cd /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping
cp *.contigs-fixed.fsa /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies

### Binning

#### Metabat2, Concoct, Maxbin2

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long # extend time limit to longer than 2 days if needed
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-binning-mmea-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate binning

#set parameters for binning:
SAMPLENAME="mmea"
READSPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data"
F_READS="DR_${SAMPLENAME}_reads_R1.fastq.gz"
R_READS="DR_${SAMPLENAME}_reads_R2.fastq.gz"
CONTIGPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies"
CONTIGFILE="${SAMPLENAME}.contigs-fixed.fsa"
BAMPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping"
METABINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/MetaBAT2_bins"
mkdir -p $METABINDIR
MAXBINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Maxbin2_bins"
mkdir -p $MAXBINDIR

#run MetaBAT2
#create depth file for MetaBat2
jgi_summarize_bam_contig_depths --outputDepth $METABINDIR/MetaBAT2_depth.txt $BAMPATH/*_MMEA_*.bam

#MetaBat2 script with verbose output, minimum length (m)(has to be >=1500) and no min bin size 
metabat2 -i $CONTIGPATH/$CONTIGFILE -a $METABINDIR/MetaBAT2_depth.txt \
-o $METABINDIR/metabat2 -m 1500

#run CheckM on metabat2 bins
checkm lineage_wf -x fa -t 3 $METABINDIR/ $METABINDIR/checkm-bins-stats

conda deactivate
echo "deactivated binning environment"

#run CONCOCT (has it's own environment)
module load conda/latest
conda activate concoct_env
echo "activated concoct environment"

CONCBINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Concoct_bins"
mkdir -p $CONCBINDIR
CTEMPDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/binning_tmp/concoct_${SAMPLENAME}_temp"
mkdir -p $CTEMPDIR

#creates the CONCOCT depth file 
cut_up_fasta.py $CONTIGPATH/$CONTIGFILE -c 10000 -o 0 --merge_last -b $CONCBINDIR/"${SAMPLENAME}"_contigs_cut.bed > $CONCBINDIR/"${SAMPLENAME}"_contigs_cut.fa

#estimate contig coverage
concoct_coverage_table.py $CONCBINDIR/"${SAMPLENAME}"_contigs_cut.bed $BAMPATH/*_MMEA_*.bam > $CONCBINDIR/coverage_table_"${SAMPLENAME}".tsv || { echo 'Exit code 2: failed to create coverage file, exiting.' && exit; }

#run CONCOCT
concoct --composition_file $CONCBINDIR/"${SAMPLENAME}"_contigs_cut.fa --coverage_file $CONCBINDIR/coverage_table_"${SAMPLENAME}".tsv -t 3 -b $CTEMPDIR || { echo 'Exit code 3: CONCOCT failed to run, exiting.' && exit; }
merge_cutup_clustering.py $CTEMPDIR/clustering_gt1000.csv > $CTEMPDIR/"${SAMPLENAME}"_clustering_merged.csv || { echo 'Exit code 4: failed to merge clusters, exiting.' && exit; }
extract_fasta_bins.py $CONTIGPATH/$CONTIGFILE $CTEMPDIR/"${SAMPLENAME}"_clustering_merged.csv --output_path $CONCBINDIR || { echo 'Exit code 5: Bins were not extracted, exiting.' && exit; }

conda deactivate
echo "deactivated conconct environment"

conda activate binning
echo "activated binning environment"

#run CheckM on CONCOCT bins
checkm lineage_wf -x fa -t 3 $CONCBINDIR/ $CONCBINDIR/checkm-bins-stats

#run Maxbin2
run_MaxBin.pl -contig $CONTIGPATH/$CONTIGFILE -reads $READSPATH/$F_READS -reads2 $READSPATH/$R_READS -out $MAXBINDIR/maxbin2 -thread 16

#run CheckM on Maxbin2 bins
checkm lineage_wf -x fasta -t 3 $MAXBINDIR/ $MAXBINDIR/checkm-bins-stats

conda deactivate

# JOB-ID: 51972917
# bash script file name: DR_binning_mmea.sh

job IDs: \
mcav: 51973345 \
ssid: 51975772 \
dcyl: 51989339 \
dlab: 51973251 \
pstr: 51975773

#### De-replicating bins with Das Tool

In [ ]:
#prep all the input files for das_tool
salloc -c 6 -p cpu
#Convert fasta output from MetaBat2, CONCOCT, and Maxbin2 into the correct format for DAS tool 
conda activate binning
Fasta_to_Contig2Bin.sh -i MetaBAT2_bins/ -e fa > ./metabat2.contigs2bin.tsv
Fasta_to_Contig2Bin.sh -i Maxbin2_bins/ -e fasta > ./maxbin2.contigs2bin.tsv
perl -pe "s/,/\tconcoct./g;" /scratch4/workspace/nikea_ulrich_uml_edu-DR_data/binning_tmp/concoct_mcav_temp/mcav_clustering_merged.csv > ./concoct.contigs2bin.tsv


#remove the first row (heading) of concoct.contigs2bin.tsv
sed -i '1d' concoct.contigs2bin.tsv

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-das_tool-mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate binning

# Set parameters
SAMPLENAME="mcav"
CONCBINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Concoct_bins/concoct.contigs2bin.tsv"  
METABINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/MetaBAT2_bins/metabat2.contigs2bin.tsv"
MAXBINDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Maxbin2_bins/maxbin2.contigs2bin.tsv"
CONTIGPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/assemblies"
CONTIGFILE="${SAMPLENAME}.contigs-fixed.fsa"
OUTDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Das_Tool"
mkdir -p $OUTDIR

#Run DAS Tool
DAS_Tool -i $CONCBINDIR,$METABINDIR,$MAXBINDIR \
-l concoct,metabat2,maxbin2 \
-c $CONTIGPATH/$CONTIGFILE \
-t 11 \
--write_bin_evals \
--write_bins \
-o $OUTDIR/"${SAMPLENAME}"

#Run CheckM
checkm lineage_wf -x fa -t 3 $OUTDIR/"${SAMPLENAME}"_DASTool_bins $OUTDIR/CheckM_stats

# -i input list: tab seperated table of contigs-bins 
#--score_threshold default is 0.5

# JOB-ID: 52026075
# bash script file name: DR_das_tool_mcav.sh

job IDs: \
mmea: 52026226 \
ssid: 52026227 \
dcyl: 52026213 \
dlab: 52026220 \
pstr: 52026228

In [ ]:
#MCAV
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  Bin Id                    Marker lineage            # genomes   # markers   # marker sets    0     1    2    3   4   5+   Completeness   Contamination   Strain heterogeneity  
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  metabat2.24         p__Cyanobacteria (UID2192)          79         584           458         9    569   6    0   0   0       98.33            0.84              16.67          
  metabat2.4            k__Bacteria (UID3187)            2258        188           117         4    180   4    0   0   0       96.58            3.42              25.00          
  metabat2.3            k__Bacteria (UID3187)            2258        188           117         6    179   3    0   0   0       94.87            2.56              33.33          
  concoct.13_sub   c__Alphaproteobacteria (UID3305)      564         349           230         88   236   25   0   0   0       74.12            9.16              44.00          
  metabat2.9           o__Rhizobiales (UID3447)          356         416           249        114   298   4    0   0   0       70.73            0.72              25.00          
  concoct.58          o__Cytophagales (UID2936)           47         454           336        164   278   12   0   0   0       62.04            2.64              25.00          
  metabat2.19           k__Bacteria (UID2570)            433         274           183         84   184   6    0   0   0       57.81            1.36               0.00          
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#MMEA
------------------------------------------------------------------------------------------------------------------------------------------------------------------
  Bin Id            Marker lineage      # genomes   # markers   # marker sets   0     1    2    3   4   5+   Completeness   Contamination   Strain heterogeneity  
------------------------------------------------------------------------------------------------------------------------------------------------------------------
  concoct.122   k__Bacteria (UID2565)      2921        152            93        15   127   10   0   0   0       88.51            5.66              90.00          
------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#SSID
----------------------------------------------------------------------------------------------------------------------------------------------------------------------
  Bin Id              Marker lineage         # genomes   # markers   # marker sets   0    1    2    3   4   5+   Completeness   Contamination   Strain heterogeneity  
----------------------------------------------------------------------------------------------------------------------------------------------------------------------
  concoct.211   p__Cyanobacteria (UID2143)      129         472           368        3   449   18   2   0   0       99.18            3.79              20.83          
  concoct.38       k__Bacteria (UID203)         5449        100            54        4    64   31   1   0   0       92.59           17.44              11.76          
----------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#DCYL
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  Bin Id              Marker lineage         # genomes   # markers   # marker sets    0     1    2    3   4   5+   Completeness   Contamination   Strain heterogeneity  
------------------------------------------------------------------------------------------------------------------------------------------------------------------------
  concoct.200   p__Cyanobacteria (UID2143)      129         472           368         97   361   14   0   0   0       80.03            3.49              35.71          
  concoct.71      k__Bacteria (UID1453)         901         171           117        107    60   4    0   0   0       21.14            0.75               0.00          
------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#DLAB
----------------------------------------------------------------------------------------------------------------------------------------------------------------
  Bin Id           Marker lineage      # genomes   # markers   # marker sets   0     1    2   3   4   5+   Completeness   Contamination   Strain heterogeneity  
----------------------------------------------------------------------------------------------------------------------------------------------------------------
  concoct.79   k__Bacteria (UID2570)      433         274           183        73   196   3   2   0   0       63.28            2.55               0.00          
----------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
#PSTR
Error:  No bins with bin-score >0.5 found. Adjust score_threshold to report bins with lower quality. 

### GTDB for MAG classification

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=150G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-gtdb_mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/gtdbtk-2.5.2

# Set parameters
SAMPLENAME="mcav"
BINPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Das_Tool/${SAMPLENAME}_DASTool_bins"
OUTDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/MAG_taxonomy/${SAMPLENAME}"
mkdir -p $OUTDIR

#Run gtdb-tk
gtdbtk classify_wf -x fa --genome_dir $BINPATH/ --out_dir $OUTDIR/gtdb_out

#This will process all genomes in the directory <my_genomes> using both bacterial and archaeal marker sets and place the results in <output_dir>.
#Genomes must be in FASTA format (gzip with the extension .gz is acceptable)

# JOB-ID: 52664359
# bash script file name: DR_gtdb_taxonomy_mcav.sh

Job IDs for the others: \
mmea: 52672201 \
ssid: 52672255 \
dcyl: 52672256 \
dlab: 52672257 \
no pstr bins

### MAG annotation with eggnog (maybe not - not working yet)

In [ ]:
##INSTALLATION
salloc -p cpu -t 3:00:00 -c 4
module load conda/latest
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/eggnog-mapper2.1.13
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/eggnog-mapper2.1.13
conda install bioconda::eggnog-mapper
    OMPI_MCA_opal_cuda_support=true
#said I should do this after install but not sure if necessary, holding off for now    
    conda install -c conda-forge ucx
    OMPI_MCA_osc="ucx
    OMPI_MCA_pml="ucx"
    UCX_MEMTYPE_CACHE=n

In [ ]:
#eggnog data on unity: /datasets/bio/eggnog6-data
EGGNOG_DATA_DIR=/datasets/bio/eggnog6-data

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH --qos=long #just in case takes longer than 2 days
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-eggnog-mcav-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/eggnog-mapper2.1.13

#add path variables here
SAMPLENAME="mcav"
DB="/datasets/bio/eggnog6-data"
MAGS="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Das_Tool/${SAMPLENAME}_DASTool_bins"
OUTDIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/eggnog_annot/${SAMPLENAME}"
mkdir -p $OUTDIR

#make a list of the MAGS
cd $MAGS
ls *.fa > "${SAMPLENAME}_mags.txt"
sed -i 's/.fa//g' "${SAMPLENAME}_mags.txt"

SAMPLELIST="${SAMPLENAME}_mags.txt"
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/binning/${SAMPLENAME}/Das_Tool/${SAMPLENAME}_DASTool_bins"

#run gene prediction on MAGs
while IFS= read -r SAMPLEID; do
emapper.py -m diamond --itype genome --genepred prodigal \
    -i $MAGS/"${SAMPLEID}".fa -o "${SAMPLEID}" --data_dir $DB --output_dir $OUTDIR
done < "$LISTPATH/${SAMPLELIST}"
echo "Annotation success!"

conda deactivate

# JOB-ID: 53180325
# bash script file name: DR_eggnog-mcav.sh

was not getting eggnog to work successfully ugh, I think BAKTA might be a better alternative and seems to be more updated w.r.t. resources.

### BAKTA for annotation
https://github.com/oschwengers/bakta

In [ ]:
## INSTALLATION
salloc -p cpu -t 3:00:00 -c 4
module load conda/latest
conda create -p /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/bakta -c conda-forge -c bioconda bakta
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/bakta
bakta --version # bakta 1.12.0

In [ ]:
#download full database
#started tmux session
bakta_db download --output /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/bakta_db --type full
#takes 30 min

bakt script here, have it cycle through each genome -double check if you need to report it's taxonomy though

--db /project/pi_sarah_gignouxwolfsohn_uml_edu/nikea/bakta_db/db

### Running Kraken2 on assembled contigs

need to convert each sample bam file into fastq - using bedtools

In [ ]:
##INSTALLATION
module load conda/latest
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/bedtools
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/bedtools
conda install bioconda::bedtools
conda install bioconda::samtools

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-bam2fastq-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/bedtools

BAMPATH="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping"
SORTEDOUT="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping/sorted_bams"
mkdir -p $SORTEDOUT
FASTQOUT="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping/aln_fastqs"
mkdir -p $FASTQOUT
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="all_sampleids.txt"
#concatented sample lists for each species

while IFS= read -r SAMPLEID; do

#sort BAM files by read name - this keeps the resulting records in the 2 output fastq files in the same order
samtools sort -n -o $SORTEDOUT/"${SAMPLEID}".sort.bam $BAMPATH/"${SAMPLEID}".bam

#create 2 fastq filed for paired-end sequences
bedtools bamtofastq -i $SORTEDOUT/"${SAMPLEID}".sort.bam -fq $FASTQOUT/"${SAMPLEID}"_aln_R1.fq -fq2 $FASTQOUT/"${SAMPLEID}"_aln_R2.fq
 if [ $? -eq 0 ]; then
        echo "completed successfully for sample: $SAMPLEID"
    else
        echo "encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Bam2fastq success!"

conda deactivate

# JOB-ID: 52432013
# bash script file name: DR_bam2fastq.sh

#### Now run Kraken2 on the fastqs

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --qos=long # extend time limit to longer than 2 days if needed
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/slurm_outs/slurm-kraken2_pluspfdb-aln-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kraken2

DBNAME="/datasets/bio/kraken2/PlusPF"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/mapping/aln_fastqs"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-DR_data/kraken2_pluspf_alns"
mkdir -p $OUTDIR
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/DR/ID_tables"
SAMPLELIST="all_sampleids.txt"

# classify each set of paired end reads against PlusPF database
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $SLURM_CPUS_ON_NODE --report $OUTDIR/"${SAMPLEID}"_aln.kreport2 --report-zero-counts --memory-mapping --paired $READS/"${SAMPLEID}"_aln_R1.fq $READS/"${SAMPLEID}"_aln_R2.fq > $OUTDIR/"${SAMPLEID}"_aln.kraken2 
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 52663237
# bash script file name: /bash_scripts/DR_kraken2_pluspfdb_aln.sh

lower classification rates than the filtered and repaired reads. \
running same script but with prackenDB just to see (52666046)